# Projekt Arbeit ML
### Elisa Sirigu, Doga Kaya, Lea Hanimann, Deborah Burri

### 1. Problemstellung

Ziel ist es, basierend auf Daten nach dem 3. Snapshot vorherzusagen, ob eine Kampagne von Encore Events voraussichtlich erfolgreich sein wird oder ob Interventionsbedarf herrscht.

### 2. Definition Klassenattribut

Wir modellieren die Aufgabe als Klassifikationsproblem.

Daher wählen wir als Klassenattribut: `needs_intervention`

- 1 = Kampagne braucht Hilfe
- 0 = Kampagne wird ohne Hilfe erfolgreich

Unser Klassenattribut `needs_intervention` wird folgendermassen berechnet:

$$
\mathrm{needs\_intervention} =
\begin{cases}
1, & \text{wenn } \mathrm{final\_impressions} < 0.9 \cdot \mathrm{target} \\
0, & \text{sonst}
\end{cases}
$$

### 3. Datenaufbereitung

#### 3.1 Daten einlesen

In [2]:
import pandas as pd

df_basic = pd.read_csv("inputfiles/campaign_basic_information.csv")
df_channels = pd.read_csv("inputfiles/campaign_channels.csv")
df_snapshots = pd.read_csv("inputfiles/campaign_snapshots.csv")

#### 3.2 Transformation der Channel und Snapshot Daten

Die Tabelle `campaign_channels` liegt im sogenannten Long-Format vor, wobei eine Kampagne mehrere Zeilen besitzen kann, da sie über mehrere Marketingkanäle ausgespielt wird.

Für Machine Learning wird jedoch eine Struktur benötigt, bei der jede Instanz genau eine Zeile repräsentiert.

Daher wird die Tabelle mittels Pivot-Transformation in ein Wide-Format überführt.

Dabei gilt:

Jeder Kanal wird zu einer eigenen Spalte
Werte sind binär (0 oder 1)

Interpretation:

- 1 = Kanal wird verwendet
- 0 = Kanal wird nicht verwendet

Diese Transformation entspricht einem Multi-Hot-Encoding und ermöglicht es, kategoriale Informationen in numerischer Form in das Modell einzubringen.

In [3]:
df_channels["value"] = 1

channels_pivot = df_channels.pivot_table(
    index="campaign_id",
    columns="channel",
    values="value",
    fill_value=0
).reset_index()

channels_pivot.columns.name = None

In [4]:
snapshots_impr = df_snapshots.pivot(
    index="campaign_id",
    columns="snapshot_number",
    values="impressions"
)
snapshots_impr.columns = [f"impressions_s{col}" for col in snapshots_impr.columns]

snapshots_days = df_snapshots.pivot(
    index="campaign_id",
    columns="snapshot_number",
    values="days_after_start"
)
snapshots_days.columns = [f"days_after_start_s{col}" for col in snapshots_days.columns]

snapshots_pivot = snapshots_impr.merge(
    snapshots_days,
    left_index=True,
    right_index=True
).reset_index()

#### 3.3 Daten zusammenführen

In [5]:
df = df_basic.merge(snapshots_pivot, on="campaign_id", how="left")
df = df.merge(channels_pivot, on="campaign_id", how="left")

#### 3.4 Fehlende Channel-Werte mit 0 auffüllen

In [6]:
channel_cols = ["facebook", "instagram", "google_search", "google_display"]

for col in channel_cols:
    if col not in df.columns:
        df[col] = 0

df[channel_cols] = df[channel_cols].fillna(0)

#### 3.5 Zielvariable erstellen

Die Zielvariable `needs_intervention` beschreibt, ob eine Kampagne voraussichtlich Unterstützung benötigt.

Sie basiert auf den finalen Impressions (4. Snapshot):

- 1 = Ziel (90%) wird nicht erreicht
- 0 = Ziel wird erreicht

In [7]:
df["needs_intervention"] = (
    df["impressions_s4"] < 0.9 * df["target"]
).astype(int)

#### 3.6 Entfernen irrelevanter Attribute

Das Attribut `campaign_id` würde unserem Modell keine generalisierbaren Muster liefern, das heisst, dass `campain_id` nur als Identifaktor dient und über keine inhaltlich relevante Bedeutung für uns verfügt.
Snapshot 4 (`final_impressions`) wird ebenfalls enfernt, da wenn man dieses Attribut im Datensatz lassen würde, würde es zu Data Leakage führen.

In [8]:
df = df.drop(columns=["campaign_id", "impressions_s4"]) # campaign_id und impressions_s4 werden nicht mehr benötigt

#### 3.7 Kontrolle

Zur Kontrolle lassen wir uns, unser bereinigtes Dataset herausgeben.

In [9]:
df

,target,budget,start_month,end_month,start_week,end_week,days,region,category,impressions_s1,...,impressions_s3,days_after_start_s1,days_after_start_s2,days_after_start_s3,days_after_start_s4,facebook,google_display,google_search,instagram,needs_intervention
0,4080000,20400,9,11,39,47,57,germany,other,5929169.0,...,9122437.0,14,28,43,57,0,0,0,1,0
1,3660000,18300,12,5,49,19,154,austria,conference,1852771.0,...,3136229.0,38,77,116,154,1,1,1,1,0
2,2180000,10900,5,6,19,24,33,switzerland,festival,851049.0,...,1593937.0,8,16,25,33,1,1,1,1,0
3,2140000,10700,3,6,9,25,110,germany,conference,1009261.0,...,1715259.0,28,55,82,110,1,1,1,1,0
4,2120000,10600,5,8,20,35,473,germany,comedy,943431.0,...,1665276.0,118,236,355,473,1,0,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
236,20000,100,10,11,43,46,18,germany,conference,2687.0,...,12337.0,4,9,14,18,1,0,1,0,1
237,20000,100,9,11,36,46,70,germany,comedy,2681.0,...,10056.0,18,35,52,70,1,0,0,1,0
238,20000,100,12,12,49,52,21,switzerland,theatre,8132.0,...,14617.0,5,10,16,21,1,0,0,0,1
239,20000,100,4,4,13,18,29,germany,festival,4548.0,...,15235.0,7,14,22,29,1,0,0,0,0


### 4. Erstellen einer CSV-Datei des bereigneten Datensatzes

In [10]:
df.to_csv("inputfiles/processed_data.csv", index=False)